# Impetus — Energy-Based Reasoning for Open LLMs
## Google Colab Notebook | Baseline Benchmarking

**Strategy:** Math + Logic first (GSM8K, ARC, BBH)

---
### Instructions
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells
3. Results saved as CSV in `/content/Impetus/results/`
4. Download from Files sidebar on the left

In [ ]:
import sys, os, subprocess
from pathlib import Path

# ── Install dependencies ──
DEPS = [
    "torch>=2.1.0",
    "transformers>=4.36.0",
    "accelerate>=0.25.0",
    "datasets>=2.16.0",
    "evaluate>=0.4.1",
    "sentencepiece>=0.1.99",
    "bitsandbytes>=0.41.0",
    "scipy",
    "pandas",
    "tqdm",
    "git+https://github.com/huggingface/transformers.git",  # bleeding edge for Qwen
]
for dep in DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", dep])

print("Dependencies installed.")

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Clone project ──
REPO_URL = "https://github.com/EdhieBM/Impetus.git"
PROJECT_DIR = "/content/Impetus"

if not os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, PROJECT_DIR])

os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print(f"Working in: {os.getcwd()}")
print(f"Contents: {os.listdir('.')}")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print("Model loaded successfully")

In [ ]:
# ── Phase 1: Baseline GSM8K ──
from benchmarks.run_baseline import run_gsm8k, save_results

print("Running GSM8K baseline (50 samples for speed)...")
results = run_gsm8k(model, tokenizer, max_samples=50)
path = save_results(results, MODEL_NAME, "results")
print(f"Saved: {path}")

In [ ]:
# ── Evaluate ──
from benchmarks.evaluate_results import evaluate_gsm8k

stats = evaluate_gsm8k(results)
print(f"GSM8K  | Acc: {stats['accuracy']:.2%}  ({stats['correct']}/{stats['total']})  | Latency: {stats['avg_latency_s']:.2f}s")

In [ ]:
import pandas as pd
print("\n=== Session Summary ===")
print(f"Model: {MODEL_NAME}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"Baseline results: {stats}")

pd.DataFrame(results).to_csv("results/colab_baseline_full.csv", index=False)
print("\nCSV saved to results/colab_baseline_full.csv")